# Energy Intelligence Platform

## Notebook 02 - RAG Chat Engine

Este notebook implementa el motor conversacional basado en Retrieval-Augmented Generation (RAG).

Componentes:

1. Carga del índice vectorial generado en Notebook 01.
2. Recuperación semántica mediante FAISS.
3. Generación de respuestas con Groq.
4. Citación de fuentes documentales.

Artefactos utilizados:

- energy_rag.index
- energy_chunks.pkl

Este notebook servirá posteriormente como base para:

- Notebook 03 (Sistema Multi-Agente)
- API FastAPI
- Integración con Lovable

## 1. Instalación

In [15]:
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q groq

## 2. Configuracion

In [16]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
BASE_PATH = "/content/drive/MyDrive/EnergyRag"

INDEX_PATH = f"{BASE_PATH}/energy_rag.index"
CHUNKS_PATH = f"{BASE_PATH}/energy_chunks.pkl"

print(INDEX_PATH)
print(CHUNKS_PATH)

/content/drive/MyDrive/EnergyRag/energy_rag.index
/content/drive/MyDrive/EnergyRag/energy_chunks.pkl


## 3. Carga de FAISS

In [18]:
import faiss

index = faiss.read_index(INDEX_PATH)

print("Vectores cargados:", index.ntotal)

Vectores cargados: 9986


## 4. Carga de Chunks

In [19]:
import pickle

with open(CHUNKS_PATH, "rb") as f:
    chunks = pickle.load(f)

print("Chunks cargados:", len(chunks))

Chunks cargados: 9986


## 5. Carga de modelo embeddings

In [20]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Modelo cargado")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Modelo cargado


## 6. Conectamos Groq

In [21]:
import os
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

print("Groq conectado")

Groq conectado


### 6.1 Funcion de busqueda semántica

In [22]:
import numpy as np

def search(query, k=5):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

### 6.2 Construccion del contexto

In [23]:
def build_context(results):

    context = "\n\n".join(
        [
            f"[SOURCE: {r['source']}]\n{r['text']}"
            for r in results
        ]
    )

    return context

### 6.3 Funcion Rag

In [24]:
def answer_question(question):

    retrieved = search(
        question,
        k=5
    )

    context = build_context(
        retrieved
    )

    prompt = f"""
You are an Energy Intelligence Assistant.

Answer ONLY using the information contained in the context.

If the answer is not present, say:
"I cannot find this information in the knowledge base."

CONTEXT:

{context}

QUESTION:

{question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    return {
        "answer": answer,
        "sources": list(
            set(
                [r["source"] for r in retrieved]
            )
        )
    }

In [25]:
# primera prueba
result = answer_question(
    "What is the role of energy storage in the energy transition?"
)

print(result["answer"])

print("\nSOURCES:\n")

for s in result["sources"]:
    print("-", s)

Energy storage, such as batteries, plays a crucial role in the energy transition by allowing for the absorption of wind power and other intermittent renewable energy sources, improving the investment case for renewables, and maintaining reliability in the power system. It helps to balance generation and consumption, reduce curtailment, and smooth peaks, thereby increasing the system's ability to integrate renewable energy sources.

SOURCES:

- 1_World_Energy_Outlook_2025.pdf
- 4_Global_Wind_Report_2026.pdf
- 12_EnergyandAI.pdf


In [26]:
# prueba España
result = answer_question(
    "What are the main objectives of Spain's PNIEC 2030?"
)

print(result["answer"])

print("\nSOURCES:\n")

for s in result["sources"]:
    print("-", s)

I cannot find this information in the knowledge base.

SOURCES:

- 13_PNIEC_España_2024_240924.pdf


In [27]:
result = answer_question(
    "How is artificial intelligence impacting electricity demand?"
)

print(result["answer"])

print("\nSOURCES:\n")

for s in result["sources"]:
    print("-", s)

Artificial intelligence (AI) is having a significant impact on electricity demand. Rapidly growing investment in data centres is already straining grids in some places and raising concerns about the ability of the electricity system to meet a surge in demand. However, the exact share of data centre electricity demand that comes from AI is challenging to answer due to the lack of comprehensive data on the share of different kinds of workloads and limited visibility over specific workloads running in data centre facilities. Additionally, electricity demand is already growing strongly in emerging market and developing economies, driven by economic growth, industrialisation, and increased adoption of appliances, which may be further exacerbated by AI adoption.

SOURCES:

- 12_EnergyandAI.pdf
